# Phase 1: Data Preprocessing for Olympic Games
Run these cells one by one (`Shift + Enter`) to see how data transforms step by step. This matches your exact problem statement for creating athlete participation counts, medal totals, and country-wise performance across years!

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Define the path to your dataset 
# (using the one you just downloaded!)
csv_path = '/Users/raj/Downloads/athlete_events.csv'

print("Libraries imported successfully!")

In [ ]:
# 1. Load the dataset using pandas
print("Loading data...")
df = pd.read_csv(csv_path)

print(f"Dataset shape: {df.shape}")
print("First 5 rows of the raw data (One row = One Athlete in one Event):")
df.head()

In [ ]:
# 2. Exploring and Cleaning Missing Values
print("Missing values in Medal column before cleaning:", df['Medal'].isnull().sum())

# Since we only care about medals won, a missing medal means 'None'
df['Medal'] = df['Medal'].fillna('None')

print("Missing values after cleaning:", df['Medal'].isnull().sum())
df[['Name', 'Event', 'Medal']].head(10)

In [ ]:
# 3. Encoding Variables (One-Hot Encoding Medals)
# This splits 'Medal' into countable dummy numerical columns
medal_dummies = pd.get_dummies(df['Medal'], prefix='Medal', dtype=int)

# Drop the 'None' column because counting non-medals is mathematically useless for totals
if 'Medal_None' in medal_dummies.columns:
    medal_dummies = medal_dummies.drop('Medal_None', axis=1)

# Append new encoded columns to dataframe
df = pd.concat([df, medal_dummies], axis=1)

print("Notice the 3 new Medal binary columns added:")
df[['Name', 'Medal', 'Medal_Bronze', 'Medal_Silver', 'Medal_Gold']].head(10)

In [ ]:
# 4. Aggregation / Transformation (The Magic Step)
# We group by Country ('NOC') and 'Year' to extract exact requirements:
# - Athlete Participation Counts
# - Event Statistics
# - Medal Totals

country_year_data = df.groupby(['NOC', 'Year']).agg(
    Athlete_Participation_Count=('ID', 'nunique'),    # Count unique athlete IDs
    Event_Count=('Event', 'nunique'),                 # Count unique events
    Total_Gold=('Medal_Gold', 'sum'),                 # Sum dummy columns
    Total_Silver=('Medal_Silver', 'sum'),
    Total_Bronze=('Medal_Bronze', 'sum')
).reset_index()

# Create final total medals feature
country_year_data['Total_Medals'] = (
    country_year_data['Total_Gold'] + 
    country_year_data['Total_Silver'] + 
    country_year_data['Total_Bronze']
)

print(f"New Aggregated Dataset Shape: {country_year_data.shape}")
print("Every row represents ONE country's overall performance in ONE specific year!")
country_year_data.head(10)

In [ ]:
# 5. Data Normalization & Scaling
# Large differences between participation counts and other metrics are smoothed out
# using standard scaling (mean 0, variance 1) which speeds up ML Models greatly.

scaler = StandardScaler()

numeric_columns = ['Athlete_Participation_Count', 'Event_Count']
country_year_data_scaled = country_year_data.copy()
country_year_data_scaled[numeric_columns] = scaler.fit_transform(country_year_data_scaled[numeric_columns])

print("Final Normalized Data for Machine Learning (Phase 1 Complete!):")
country_year_data_scaled.head(10)